## Clinical Information Extraction with LLMs

This notebook demonstrates a pipeline for extracting structured clinical information
from emergency room (ER) reports using a large language model. It extracts:
- precipitating event (cause of ER visit)
- confidence
- explanation

### Pipeline

1. Clinical text extraction
2. Prompt-based information extraction
3. Structured JSON parsing

In [1]:
import json

from src.data_loader.preprocessing import ClinicalPreprocessor
from src.information_extraction.extractor import MedicalExtractor


In [2]:
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

extractor = MedicalExtractor(MODEL_NAME)


Loading model: mistralai/Mistral-7B-Instruct-v0.3...


/home/tatiana.bladier/.local/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


In [3]:
en_markers = ["CHIEF COMPLAINT:", 
            "HISTORY OF PRESENT ILLNESS:", 
            "REASON FOR VISIT:"]

en_stop_marker = "About This Sample:"

en_preprocessor = ClinicalPreprocessor(
    language="en", 
    markers=en_markers, 
    end_marker=en_stop_marker
)

In [9]:
# ── 2. Prompt ─────────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are a medical information extraction assistant.
Given a medical text, extract structured information and return it as JSON only,
with no preamble or explanation.
"""

USER_INSTRUCTION_TEMPLATE = """Identify the precipitating event that led to the medical emergency.

Return ONLY JSON with these keys:
- event (string)
- confidence (float 0-1)
- explanation (string)

Medical text:
{text}"""

In [10]:
# Prompt example

def build_prompt(text: str) -> list:
    """
    Constructs a structured chat-style prompt for extracting medical events.
    Returns a list of messages (system and user) for the model.
    """
    user_content = USER_INSTRUCTION_TEMPLATE.format(text=text)
    return [
        {
            "role": "system", 
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user", 
            "content": user_content
        }
    ]

In [ ]:
en_triage_notes_raw = "data/raw_data/mtsamples_er/texts"

df_en = en_preprocessor.process_folder(en_triage_notes_raw, extractor_instance=extractor, limit=10, prompt_func=build_prompt)
df_en

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [7]:
df_en["event"].value_counts()

event
Abdominal pain                                                   2
Fell off bicycle                                                 2
Accidental ingestion of Celesta                                  2
Family believes the patient was poisoned at her care facility    2
Acute Inferior Myocardial Infarction                             1
Obtundation and free air under the right diaphragm               1
Name: count, dtype: int64

In [8]:
for _, row in df_en.iterrows():
    print(f"--- {row['source_file']} ---")
    print(json.dumps(row.to_dict(), indent=2))

--- Abdominal_Pain_-_Consult.txt ---
{
  "event": "Abdominal pain",
  "confidence": 0.95,
  "explanation": "The patient presented with a persistent abdominal pain for approximately 7-8 days. The pain is localized to the left lower quadrant.",
  "source_file": "Abdominal_Pain_-_Consult.txt",
  "lang": "en"
}
--- Abdominal_Pain_2D_Consult.txt ---
{
  "event": "Abdominal pain",
  "confidence": 0.95,
  "explanation": "The patient presented with a persistent abdominal pain for approximately 7-8 days. The pain is localized to the left lower quadrant.",
  "source_file": "Abdominal_Pain_2D_Consult.txt",
  "lang": "en"
}
--- Abrasions_26_Lacerations_2D_ER_Visit.txt ---
{
  "event": "Fell off bicycle",
  "confidence": 0.95,
  "explanation": "The precipitating event that led to the medical emergency was the patient falling off his bicycle, as stated in the History of Present Illness section.",
  "source_file": "Abrasions_26_Lacerations_2D_ER_Visit.txt",
  "lang": "en"
}
--- Abrasions__Lacerations

### Evaluation